In [6]:
from pathlib import Path
from metasmith.python_api import Agent, Source, SshSource, Std, DataInstanceLibrary, WorkflowTask
from local.constants import WORKSPACE_ROOT

dtypes, containers, transforms = Std()

agent_home = Source.FromLocal(Path("./cache/local_home").resolve())
# agent_home = SshSource("fir", Path("/scratch/phyberos/metasmith")).AsSource()
smith = Agent(
    home = agent_home,
)
# smith.Deploy()

In [10]:
[k for k in dtypes.types if "reads" in k]

['long_reads',
 'hifi_reads',
 'nanopore_reads',
 'long_reads_accession',
 'long_reads_assembly',
 'long_reads_filtered',
 'reads',
 'paired_reads_forward',
 'paired_reads_reverse',
 'short_reads',
 'short_reads_accession',
 'short_reads_assembly',
 'short_reads_trimmed']

In [7]:
inputs = DataInstanceLibrary("./cache/pointed_reads.xgdb")
inputs.AddItem(WORKSPACE_ROOT/"main/local_mock/cache/flye_in/lr_ss10.fastq", "std::hifi_reads")
inputs.AddItem(WORKSPACE_ROOT/"main/local_mock/cache/asm.xgdb/scadc.fna", "std::assembly")
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)


# remote_root = Path("/project/6004975/phyberos/cyanoverse/main/logistics/interleave_test.xgdb")
# inputs = DataInstanceLibrary("./cache/remote_test.xgdb")
# inputs.AddItem(remote_root/"fwd.fq", "std::paired_reads_forward")
# inputs.AddItem(remote_root/"rev.fq", "std::paired_reads_reverse")
# inputs.Save()
# for p, n, e in inputs.Iterate():
#     print(n, p, e, e.parents)

std::hifi_reads /home/tony/workspace/tools/Metasmith/main/local_mock/cache/flye_in/lr_ss10.fastq <{data:Long sequence,format:Sequence file,read_quality:hifi}:jNuQqT4m> set()
std::assembly /home/tony/workspace/tools/Metasmith/main/local_mock/cache/asm.xgdb/scadc.fna <{data:Sequence assembly}:BD4mNSAh> set()


In [12]:
[k for k in dtypes.types if "cov" in k]

['per_contig_coverage', 'per_bp_coverage']

In [13]:
_tasks = [
    smith.GenerateWorkflow(
        given      = [containers, inputs],
        transforms = [transforms],
        targets    = [dtypes[t]]
    )
    # for t in ["short_reads"]
    for t in ["per_contig_coverage"]
]

task = WorkflowTask.Merge((_tasks))
print(task.GetKey())
for step in [s for p in task.plans for s in p.steps]:
    print(step.order, step.transform.name)
# task.RenderDAG("./cache/dag")

UV5YB6So
1 minimap
2 bedtools_genomcov


In [14]:
with open("./cache/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()
task.config = dict(
    nextflow = dict(
        preset="slurm",
        slurm_account=SLURM_ACCOUNT,
        cpus=1,
        queueSize=200,
        memory='8 GB',
        time='3h',
    )
)

In [15]:
smith.StageWorkflow(task, on_exist="clear")

2025-10-22_13-25-47  | connecting to deployed agent
2025-10-22_13-25-47  | starting relay service
 | > relay server already running at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/relay/0x155d730db2]
2025-10-22_13-25-48 W| task already staged at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/UV5YB6So]
2025-10-22_13-25-48 W| clearing previously staged task
2025-10-22_13-25-48  | sending metadata for workflow [UV5YB6So]
2025-10-22_13-25-50  | staging
 | > including dev binds
 | > 2025-10-22_13-25-51  | api call to [stage_workflow] with [{'task_key': 'UV5YB6So'}]
 | > 2025-10-22_13-25-52  | staging workflow [UV5YB6So] with [2] data libs and [1] transform libs
 | > 2025-10-22_13-25-52  | ex| /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
 | > 2025-10-22_13-25-52  | ex| /home/tony/workspace/tools/Metasmith/main/local_mock/cache/flye_in/lr_ss10.fastq
 | > 2025-10-22_13-25-52  | ex| /home/tony/workspace/tools/Metasm

In [16]:
smith.RunWorkflow(task)

2025-10-22_13-25-55  | connecting to deployed agent
2025-10-22_13-25-55  | starting relay service
 | > relay server already running at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/relay/0x155d730db2]
2025-10-22_13-25-56  | triggering execution of [UV5YB6So]
2025-10-22_13-25-57  | closing connection


In [27]:
bool("True".title()=="True")

True

In [26]:
f"{True}"

'True'

In [7]:
smith.CheckWorkflow(task)

2025-10-22_11-49-19  | connecting to deployed agent
2025-10-22_11-49-19  | starting relay service
 | > relay server already running at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/relay/0x155d730db2]
 | > including dev binds
 | > 2025-10-22_11-49-21  | api call to [check_workflow] with [{'key': 'UV5YB6So'}]
 | > 2025-10-22_11-49-21  | searching for logs
 | > 2025-10-22_11-49-21  | found [1] runs
 | > 2025-10-22_11-49-21  |     1: [logs.2025-10-22_11-49-19]
 | > 2025-10-22_11-49-21  | here is the main log of the latest run [logs.2025-10-22_11-49-19]
 | > 2025-10-22_11-49-21  | >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
 | > 2025-10-22_11-49-21  | 
 | > including dev binds
 | > 2025-10-22_11-49-20  | api call to [run_workflow] with [{'key': 'UV5YB6So', 'log_dir': '_metasmith/logs.2025-10-22_11-49-19'}]
 | > 2025-10-22_11-49-20  | start time [2025-10-22_11-49-19]
 | > 2025-10-22_11-49-20  | running workflow [UV5YB6So] with preset [default]
